# EDA 3: Feature-Target Correlations (Information Coefficient Analysis)

This notebook computes the Information Coefficient (IC) for all features:
- IC = Rank correlation between feature values and forward returns
- IC stability over time (rolling windows)
- IC by market regime
- IC decay (how quickly signal degrades)

**Goal**: Identify the most predictive features and understand their properties.

In [ ]:
import sys
sys.path.insert(0, '/home/nock/projects/quant_suite')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
from pathlib import Path
from collections import defaultdict

from src.data.sources.yahoo import YahooDataSource
from src.data.features import FeatureEngine
from src.data.feature_engineering.feature_registry import FEATURE_REGISTRY, FeatureCategory

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]

## 1. Information Coefficient Functions

Define functions for computing IC and related metrics.

In [ ]:
def compute_ic(feature: pd.Series, forward_returns: pd.Series) -> float:
    """Compute Information Coefficient (rank correlation)."""
    # Align and drop NaN
    aligned = pd.concat([feature, forward_returns], axis=1).dropna()
    if len(aligned) < 30:
        return np.nan
    return stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])[0]


def compute_rolling_ic(feature: pd.Series, forward_returns: pd.Series, window: int = 252) -> pd.Series:
    """Compute rolling IC over time."""
    aligned = pd.concat([feature, forward_returns], axis=1).dropna()
    aligned.columns = ['feature', 'returns']
    
    def rolling_spearman(x):
        if len(x) < window // 2:
            return np.nan
        return stats.spearmanr(x['feature'], x['returns'])[0]
    
    rolling_ic = aligned.rolling(window).apply(
        lambda x: rolling_spearman(aligned.loc[x.index]),
        raw=False
    )['feature']
    
    return rolling_ic


def compute_ic_decay(feature: pd.Series, prices: pd.Series, horizons: list = None) -> dict:
    """Compute IC across multiple forward horizons to see decay."""
    horizons = horizons or [1, 2, 3, 5, 10, 21, 42, 63]
    ic_decay = {}
    for h in horizons:
        fwd_return = prices.shift(-h) / prices - 1
        ic_decay[h] = compute_ic(feature, fwd_return)
    return ic_decay


def compute_ic_by_regime(feature: pd.Series, forward_returns: pd.Series, 
                         regime: pd.Series) -> dict:
    """Compute IC separately for each regime."""
    ic_by_regime = {}
    for regime_name in regime.dropna().unique():
        mask = regime == regime_name
        ic_by_regime[regime_name] = compute_ic(feature[mask], forward_returns[mask])
    return ic_by_regime


def ic_t_stat(ic: float, n: int) -> float:
    """Compute t-statistic for IC significance."""
    if np.isnan(ic) or n < 30:
        return np.nan
    return ic * np.sqrt(n - 2) / np.sqrt(1 - ic**2)


def ic_grade(ic: float) -> str:
    """Grade IC quality."""
    abs_ic = abs(ic) if not np.isnan(ic) else 0
    if abs_ic >= 0.05:
        return 'A (Excellent)'
    elif abs_ic >= 0.03:
        return 'B (Good)'
    elif abs_ic >= 0.02:
        return 'C (Acceptable)'
    elif abs_ic >= 0.01:
        return 'D (Weak)'
    else:
        return 'F (Noise)'

print("IC computation functions defined.")

## 2. Load Data and Compute Features

Load price data and compute all technical features.

In [ ]:
# Load price data
SYMBOLS = ['SPY', 'QQQ', 'IWM', 'AAPL', 'MSFT', 'NVDA', 'XLE', 'XLF']

yahoo = YahooDataSource()
price_data = {}
feature_data = {}

for symbol in SYMBOLS:
    try:
        df = yahoo.get_historical_data(symbol, period="5y")
        price_data[symbol] = df
        
        # Compute all features
        featured = FeatureEngine.add_all_features(df.copy())
        feature_data[symbol] = featured
        
        print(f"Loaded {symbol}: {len(df)} rows, {len(featured.columns) - len(df.columns)} features")
    except Exception as e:
        print(f"Failed to load {symbol}: {e}")

In [ ]:
# Get list of feature columns
if 'SPY' in feature_data:
    base_cols = ['open', 'high', 'low', 'close', 'volume', 'adj_close']
    feature_cols = [c for c in feature_data['SPY'].columns if c not in base_cols]
    print(f"Total features: {len(feature_cols)}")
    
    # Group features
    feature_groups = defaultdict(list)
    for col in feature_cols:
        prefix = col.split('_')[0]
        feature_groups[prefix].append(col)
    
    print("\nFeatures by group:")
    for group, cols in sorted(feature_groups.items()):
        print(f"  {group}: {len(cols)}")

## 3. Compute IC for All Features

Calculate IC for each feature against multiple forward return horizons.

In [ ]:
# Compute IC for all features across symbols
ic_results = []
horizons = [1, 5, 21]

for symbol in price_data.keys():
    df = feature_data[symbol]
    close = df['close']
    
    for feature_name in feature_cols:
        feature = df[feature_name]
        
        for horizon in horizons:
            fwd_return = close.shift(-horizon) / close - 1
            ic = compute_ic(feature, fwd_return)
            n = len(pd.concat([feature, fwd_return], axis=1).dropna())
            
            ic_results.append({
                'Symbol': symbol,
                'Feature': feature_name,
                'Horizon': horizon,
                'IC': ic,
                'Abs_IC': abs(ic) if not np.isnan(ic) else 0,
                'N': n,
                't_stat': ic_t_stat(ic, n),
                'Grade': ic_grade(ic)
            })

ic_df = pd.DataFrame(ic_results)
print(f"Computed {len(ic_df)} IC values")

In [ ]:
# Aggregate IC across symbols
ic_agg = ic_df.groupby(['Feature', 'Horizon']).agg({
    'IC': ['mean', 'std', 'count'],
    'Abs_IC': 'mean'
}).reset_index()
ic_agg.columns = ['Feature', 'Horizon', 'IC_mean', 'IC_std', 'Count', 'Abs_IC_mean']

# IC Information Ratio (mean / std)
ic_agg['IC_IR'] = ic_agg['IC_mean'] / ic_agg['IC_std']

# Grade based on mean IC
ic_agg['Grade'] = ic_agg['Abs_IC_mean'].apply(lambda x: ic_grade(x))

ic_agg.head(20)

## 4. Top Features by IC

Identify the most predictive features.

In [ ]:
# Top features for 5-day returns
top_features_5d = ic_agg[ic_agg['Horizon'] == 5].nlargest(20, 'Abs_IC_mean')[[
    'Feature', 'IC_mean', 'IC_std', 'IC_IR', 'Grade'
]]

print("Top 20 Features for 5-Day Forward Returns:")
top_features_5d

In [ ]:
# Visualize top features IC
fig, ax = plt.subplots(figsize=(14, 8))

top_20 = top_features_5d.head(20)
colors = ['green' if x > 0 else 'red' for x in top_20['IC_mean']]

bars = ax.barh(range(len(top_20)), top_20['IC_mean'], color=colors, alpha=0.7)
ax.errorbar(top_20['IC_mean'], range(len(top_20)), xerr=top_20['IC_std'], 
           fmt='none', color='black', capsize=3, alpha=0.5)

ax.set_yticks(range(len(top_20)))
ax.set_yticklabels(top_20['Feature'])
ax.axvline(0, color='black', linewidth=0.5)
ax.axvline(0.03, color='gray', linestyle='--', alpha=0.5, label='Good threshold (0.03)')
ax.axvline(-0.03, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Information Coefficient')
ax.set_title('Top 20 Features by IC (5-Day Forward Returns)')
ax.legend()
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# IC heatmap by horizon
ic_pivot = ic_agg.pivot(index='Feature', columns='Horizon', values='IC_mean')

# Select top 30 features by average absolute IC
top_features = ic_agg.groupby('Feature')['Abs_IC_mean'].mean().nlargest(30).index
ic_pivot_top = ic_pivot.loc[top_features]

fig, ax = plt.subplots(figsize=(8, 12))
sns.heatmap(ic_pivot_top, annot=True, cmap='RdYlBu_r', center=0, ax=ax, fmt='.3f')
ax.set_title('IC Heatmap: Top 30 Features by Horizon')
plt.tight_layout()
plt.show()

## 5. IC Stability Analysis

Analyze how stable IC is over time.

In [ ]:
# Rolling IC for top features
spy_df = feature_data['SPY']
spy_fwd_5d = spy_df['close'].shift(-5) / spy_df['close'] - 1

# Select top 5 features
top_5_features = top_features_5d.head(5)['Feature'].tolist()

# Compute rolling IC
rolling_ic_data = {}
window = 252  # 1 year

for feature_name in top_5_features:
    feature = spy_df[feature_name]
    
    # Simple rolling rank correlation
    aligned = pd.concat([feature, spy_fwd_5d], axis=1).dropna()
    aligned.columns = ['feature', 'returns']
    
    # Compute rolling correlation
    rolling_ic = []
    for i in range(window, len(aligned)):
        window_data = aligned.iloc[i-window:i]
        ic = stats.spearmanr(window_data['feature'], window_data['returns'])[0]
        rolling_ic.append((aligned.index[i], ic))
    
    rolling_ic_data[feature_name] = pd.DataFrame(rolling_ic, columns=['date', 'ic']).set_index('date')['ic']

print("Rolling IC computed for top features.")

In [ ]:
# Plot rolling IC
fig, ax = plt.subplots(figsize=(14, 6))

for feature_name, rolling_ic in rolling_ic_data.items():
    ax.plot(rolling_ic.index, rolling_ic.values, label=feature_name, alpha=0.7)

ax.axhline(0, color='black', linewidth=0.5)
ax.axhline(0.03, color='gray', linestyle='--', alpha=0.3)
ax.axhline(-0.03, color='gray', linestyle='--', alpha=0.3)
ax.set_ylabel('Rolling IC (252-day)')
ax.set_title('Rolling IC for Top Features (SPY, 5-day returns)')
ax.legend(loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# IC stability metrics
ic_stability = []
for feature_name, rolling_ic in rolling_ic_data.items():
    ic_stability.append({
        'Feature': feature_name,
        'IC_mean': rolling_ic.mean(),
        'IC_std': rolling_ic.std(),
        'IC_IR': rolling_ic.mean() / rolling_ic.std() if rolling_ic.std() > 0 else 0,
        'Pct_positive': (rolling_ic > 0).mean() * 100,
        'Pct_significant': (abs(rolling_ic) > 0.02).mean() * 100
    })

ic_stability_df = pd.DataFrame(ic_stability)
ic_stability_df.round(4)

## 6. IC Decay Analysis

Understand how IC decays over longer horizons.

In [ ]:
# IC decay for top features
decay_horizons = [1, 2, 3, 5, 10, 21, 42, 63]

ic_decay_data = []
for feature_name in top_5_features:
    feature = spy_df[feature_name]
    decay = compute_ic_decay(feature, spy_df['close'], decay_horizons)
    for h, ic in decay.items():
        ic_decay_data.append({
            'Feature': feature_name,
            'Horizon': h,
            'IC': ic
        })

ic_decay_df = pd.DataFrame(ic_decay_data)

# Plot decay curves
fig, ax = plt.subplots(figsize=(10, 6))

for feature_name in top_5_features:
    decay = ic_decay_df[ic_decay_df['Feature'] == feature_name]
    ax.plot(decay['Horizon'], decay['IC'], marker='o', label=feature_name)

ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('Information Coefficient')
ax.set_title('IC Decay Curves')
ax.legend()

plt.tight_layout()
plt.show()

## 7. IC by Volatility Regime

Analyze how IC varies across market regimes.

In [ ]:
# Define volatility regime
spy_df['returns'] = spy_df['close'].pct_change()
spy_df['vol_21d'] = spy_df['returns'].rolling(21).std() * np.sqrt(252)
spy_df['vol_percentile'] = spy_df['vol_21d'].rolling(252).rank(pct=True)
spy_df['vol_regime'] = pd.cut(
    spy_df['vol_percentile'],
    bins=[0, 0.33, 0.67, 1.0],
    labels=['Low', 'Medium', 'High']
)

# IC by regime for top features
ic_regime_data = []
for feature_name in top_5_features:
    feature = spy_df[feature_name]
    ic_by_regime = compute_ic_by_regime(feature, spy_fwd_5d, spy_df['vol_regime'])
    for regime, ic in ic_by_regime.items():
        ic_regime_data.append({
            'Feature': feature_name,
            'Regime': regime,
            'IC': ic
        })

ic_regime_df = pd.DataFrame(ic_regime_data)
ic_regime_pivot = ic_regime_df.pivot(index='Feature', columns='Regime', values='IC')
ic_regime_pivot

In [ ]:
# Visualize IC by regime
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(top_5_features))
width = 0.25

for i, regime in enumerate(['Low', 'Medium', 'High']):
    values = [ic_regime_pivot.loc[f, regime] if f in ic_regime_pivot.index else 0 for f in top_5_features]
    ax.bar(x + i * width, values, width, label=f'{regime} Vol', alpha=0.7)

ax.set_xticks(x + width)
ax.set_xticklabels(top_5_features, rotation=45, ha='right')
ax.axhline(0, color='black', linewidth=0.5)
ax.set_ylabel('IC')
ax.set_title('IC by Volatility Regime')
ax.legend()

plt.tight_layout()
plt.show()

## 8. Feature Category Analysis

Compare IC across feature categories.

In [ ]:
# Map features to categories based on prefix
category_map = {
    'return': 'Returns',
    'log': 'Returns',
    'volatility': 'Volatility',
    'parkinson': 'Volatility',
    'sma': 'Moving Average',
    'ema': 'Moving Average',
    'rsi': 'Momentum',
    'macd': 'Momentum',
    'stoch': 'Momentum',
    'williams': 'Momentum',
    'roc': 'Momentum',
    'obv': 'Volume',
    'volume': 'Volume',
    'vwap': 'Volume',
    'mfi': 'Volume',
    'bb': 'Volatility',
    'atr': 'Volatility',
    'adx': 'Trend',
    'body': 'Price Pattern',
    'upper': 'Price Pattern',
    'lower': 'Price Pattern',
    'gap': 'Price Pattern',
    'higher': 'Price Pattern',
    'dist': 'Price Pattern'
}

def get_category(feature_name):
    prefix = feature_name.split('_')[0]
    return category_map.get(prefix, 'Other')

# Add category to IC results
ic_agg['Category'] = ic_agg['Feature'].apply(get_category)

# Aggregate by category
category_ic = ic_agg[ic_agg['Horizon'] == 5].groupby('Category').agg({
    'Abs_IC_mean': 'mean',
    'IC_std': 'mean',
    'Feature': 'count'
}).rename(columns={'Feature': 'N_Features', 'Abs_IC_mean': 'Mean_Abs_IC'}).round(4)

category_ic.sort_values('Mean_Abs_IC', ascending=False)

In [ ]:
# Visualize category IC
fig, ax = plt.subplots(figsize=(10, 6))

category_ic_sorted = category_ic.sort_values('Mean_Abs_IC', ascending=True)
ax.barh(category_ic_sorted.index, category_ic_sorted['Mean_Abs_IC'], color='steelblue', alpha=0.7)
ax.axvline(0.02, color='gray', linestyle='--', alpha=0.5, label='Acceptable threshold (0.02)')
ax.set_xlabel('Mean Absolute IC')
ax.set_title('Average IC by Feature Category (5-day returns)')
ax.legend()

plt.tight_layout()
plt.show()

## 9. Feature Correlation Matrix

Check for redundancy among top features.

In [ ]:
# Correlation among top features
top_20_features = top_features_5d.head(20)['Feature'].tolist()
feature_corr = spy_df[top_20_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(feature_corr, annot=True, cmap='RdYlBu_r', center=0, ax=ax, fmt='.2f', 
           annot_kws={'size': 8})
ax.set_title('Correlation Matrix: Top 20 Features')
plt.tight_layout()
plt.show()

In [ ]:
# Identify highly correlated feature pairs
high_corr_pairs = []
for i, feat1 in enumerate(top_20_features):
    for feat2 in top_20_features[i+1:]:
        corr = feature_corr.loc[feat1, feat2]
        if abs(corr) > 0.7:
            high_corr_pairs.append({
                'Feature 1': feat1,
                'Feature 2': feat2,
                'Correlation': corr
            })

if high_corr_pairs:
    print("Highly correlated feature pairs (|r| > 0.7):")
    pd.DataFrame(high_corr_pairs).sort_values('Correlation', key=abs, ascending=False)
else:
    print("No highly correlated pairs found (|r| > 0.7)")

## 10. Summary & Recommendations

### Key Findings

1. **Top IC features**: Momentum and volatility-related features tend to have highest IC
2. **IC stability**: Most features show time-varying IC - regime-conditional models may help
3. **IC decay**: Signals decay over 10-20 days on average
4. **Regime dependency**: Some features work better in high/low volatility
5. **Redundancy**: Several features are highly correlated - consider feature selection

### Recommendations

1. Focus on features with IC > 0.02 and IC_IR > 0.5
2. Use regime-conditional models or include regime as a feature
3. Implement feature selection to remove redundant features
4. Consider shorter horizons (1-5 days) where signals are strongest
5. Ensemble models across multiple features to improve stability

In [ ]:
# Save summary
import json

# Top features summary
top_features_summary = ic_agg[ic_agg['Horizon'] == 5].nlargest(30, 'Abs_IC_mean')[[
    'Feature', 'IC_mean', 'IC_std', 'IC_IR', 'Grade'
]].to_dict('records')

summary = {
    'timestamp': datetime.now().isoformat(),
    'symbols_analyzed': list(price_data.keys()),
    'total_features': len(feature_cols),
    'horizons': horizons,
    'top_features_5d': top_features_summary,
    'category_ic': category_ic.to_dict(),
    'ic_stability': ic_stability_df.to_dict('records'),
    'recommendations': [
        'Focus on features with IC > 0.02 and IC_IR > 0.5',
        'Use regime-conditional models',
        'Implement feature selection for redundancy',
        'Consider 1-5 day horizons for stronger signals',
        'Ensemble across features for stability'
    ]
}

output_path = Path("/home/nock/quant_results/live/research/feature_ic_analysis.json")
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"Summary saved to {output_path}")